# 01 — Embeddings & the Resume Parser

Prototype two things here, then move the working code into `src/`:

1. **Resume parser** — load a resume with `src.parsing.loader.load_text`, send it to the LLM with a strict prompt, get back clean JSON (name, skills, experience, education, target_role). Validate the JSON. Move it to `src/parsing/resume_parser.py`.
2. **Embeddings** — embed a few short texts with the Gemini embedding model, compute cosine similarity between related vs unrelated sentences, and confirm similar meaning gives a higher score. Move the embedding call to `src/search/embed.py`.

Load your API key from `.env` first (`python-dotenv`).

In [11]:
# ============================================================
# SMART HIRE - NOTEBOOK 01
# Resume Parser using TypedDict / Structured Output
# ============================================================

import sys
import os
import json
from pathlib import Path
from typing import TypedDict, List

# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

# Add SmartHire-GenAI to Python path so "src" can be imported
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT)


# ------------------------------------------------------------
# 2. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

from dotenv import load_dotenv
from google import genai
from google.genai import types

from src.parsing.loader import load_text


# ------------------------------------------------------------
# 3. LOAD API KEY FROM .env.example
# ------------------------------------------------------------

env_file = PROJECT_ROOT / ".env.example"

load_dotenv(env_file)

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found in .env.example"
    )

#print("API key loaded successfully.")


# ------------------------------------------------------------
# 4. CONNECT TO GEMINI
# ------------------------------------------------------------

client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini client connected.")


# ------------------------------------------------------------
# 5. DEFINE TYPEDDICT STRUCTURE
# ------------------------------------------------------------

class Education(TypedDict):
    degree: str
    institute: str
    year: str
    score: str


class Experience(TypedDict):
    company: str
    role: str
    duration: str
    highlights: List[str]


class Resume(TypedDict):
    name: str
    skills: List[str]
    experience: List[Experience]
    education: List[Education]
    target_role: str


# ------------------------------------------------------------
# 6. STRICT AI PROMPT
# ------------------------------------------------------------

SYSTEM_PROMPT = """
You are an expert resume parser.

Extract information ONLY from the resume provided.

Do NOT invent, guess, or assume information.

Return the information using exactly these fields:

name
skills
experience
education
target_role

For experience, extract:

company
role
duration
highlights

For education, extract:

degree
institute
year
score

Rules:

1. Extract only information explicitly present in the resume.
2. Do not invent missing information.
3. If a string value is missing, return an empty string.
4. If a list value is missing, return an empty list.
5. skills must contain the skills explicitly mentioned.
6. experience must contain the candidate's work experience.
7. education must contain the candidate's education.
8. target_role should be based only on the candidate's resume.
9. Return only structured JSON matching the Resume schema.
"""


# ------------------------------------------------------------
# 7. RESUME PARSER USING TYPEDDICT
# ------------------------------------------------------------

def parse_resume(text: str) -> dict:

    response = client.models.generate_content(
        model=MODEL,
        contents=f"""
{SYSTEM_PROMPT}

RESUME:
{text}
""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Resume,
            temperature=0.2,
            max_output_tokens=2048
        )
    )

    return json.loads(response.text)


# ------------------------------------------------------------
# 8. FIND ALL RESUMES
# ------------------------------------------------------------

resume_folder = PROJECT_ROOT / "data" / "resumes"

if not resume_folder.exists():
    raise FileNotFoundError(
        f"Resume folder not found: {resume_folder}"
    )

resume_files = [
    file
    for file in resume_folder.iterdir()
    if file.suffix.lower() in [".pdf", ".docx", ".txt", ".md"]
]

#print("\nResumes found:", len(resume_files))

for file in resume_files:
    print("-", file.name)


# ------------------------------------------------------------
# 9. LOAD AND PARSE ALL THREE RESUMES
# ------------------------------------------------------------

parsed_resumes = {}

for file in resume_files:

    #print("\n" + "=" * 70)
    print("PROCESSING:", file.name)
    #print("=" * 70)

    try:

        # Read resume using your existing loader.py
        resume_text = load_text(file)

        #print("Resume loaded successfully.")
        #print("Characters extracted:", len(resume_text))

        # Parse using TypedDict structured output
        parsed_data = parse_resume(resume_text)

        parsed_resumes[file.name] = parsed_data

        #print("\nSTRUCTURED OUTPUT:")
        print(
            json.dumps(
                parsed_data,
                indent=2,
                ensure_ascii=False
            )
        )

    except Exception as e:

        print("\nERROR:")
        print(e)


# ------------------------------------------------------------
# 10. VALIDATE REQUIRED PROJECT HEADINGS
# ------------------------------------------------------------

required_fields = {
    "name",
    "skills",
    "experience",
    "education",
    "target_role"
}

#print("\n" + "=" * 70)
#print("VALIDATION")
#print("=" * 70)

for filename, data in parsed_resumes.items():

    missing_fields = required_fields - set(data.keys())

    if missing_fields:
        print(
            f"❌ {filename} missing fields: "
            f"{missing_fields}"
        )
    else:
        print(
            f"✅ {filename} contains all required fields."
        )


# ------------------------------------------------------------
# 11. DISPLAY RESUME SUMMARY
# ------------------------------------------------------------

#print("\n" + "=" * 70)
#print("RESUME SUMMARY")
#print("=" * 70)

for filename, data in parsed_resumes.items():

    print("\nResume:", filename)
    print("Name:", data.get("name"))
    print("Skills:", data.get("skills"))
    print("Experience:", data.get("experience"))
    print("Education:", data.get("education"))
    print("Target Role:", data.get("target_role"))


#print("\n" + "=" * 70)
#print("RESUME PARSER COMPLETED")
#print("=" * 70)

Project folder: c:\Users\koppa\SmartHire-GenAI
Gemini client connected.
- 01_Bhawana_Aggarwal_Data_Science.pdf
- Ananya_Sharma_Resume.pdf
- Rahul_Verma_Resume.pdf
PROCESSING: 01_Bhawana_Aggarwal_Data_Science.pdf
{
  "name": "BHAWANA AGGARWAL",
  "skills": [
    "Python",
    "C",
    "NumPy",
    "Pandas",
    "Seaborn",
    "Matplotlib",
    "Cufflinks",
    "KNN",
    "Decision Tree",
    "Linear & Logistic Regression",
    "SVM",
    "K-Means",
    "Neural Networks",
    "TensorFlow",
    "Keras",
    "RNN/LSTM",
    "NLP",
    "NER",
    "Regex",
    "SQL",
    "Oracle",
    "GCP",
    "Linux",
    "Windows"
  ],
  "experience": [
    {
      "company": "Wipro Technologies",
      "role": "Data Science / Machine Learning",
      "duration": "2 years",
      "highlights": [
        "Developed Python programs and worked on Wipro Neural Intelligence Platform.",
        "Built NLP-based entity extraction, classification and NER components using Regex, NLU APIs, Scikit-learn, Keras and 

In [24]:
# ============================================================
# SMART HIRE - NOTEBOOK 01
# EMBEDDINGS USING ACTUAL JOBS AND CAREER NOTES
# ============================================================

import sys
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT)


# ------------------------------------------------------------
# 2. IMPORT EMBEDDING FUNCTIONS FROM src/search/embed.py
# ------------------------------------------------------------

from src.search.embed import embed_text, cosine_similarity
from src import config

print("Embedding functions imported successfully.")


# ============================================================
# PART A - LOAD ACTUAL JOBS
# ============================================================

jobs_file = config.JOBS_CSV

print("\nJobs file:")
print(jobs_file)


# Load jobs CSV
jobs_df = pd.read_csv(jobs_file)

print("\nNumber of jobs:", len(jobs_df))
print("Job columns:")
print(list(jobs_df.columns))


# ------------------------------------------------------------
# 3. CREATE TEXT FOR EACH JOB
# ------------------------------------------------------------

job_texts = []

for _, row in jobs_df.iterrows():

    parts = []

    for column in config.JOB_TEXT_COLUMNS:

        if column in jobs_df.columns:

            value = row[column]

            if pd.notna(value):
                parts.append(str(value))

    job_text = "\n".join(parts)

    job_texts.append(job_text)


print("\nJobs converted to text:", len(job_texts))


# ------------------------------------------------------------
# 4. SHOW ONE REAL JOB
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SAMPLE JOB TEXT")
print("=" * 60)

print(job_texts[0][:1500])


# ============================================================
# PART B - LOAD ACTUAL CAREER NOTES
# ============================================================

career_notes_folder = config.CAREER_NOTES_DIR

print("\n" + "=" * 60)
print("CAREER NOTES")
print("=" * 60)

career_note_files = [
    file
    for file in career_notes_folder.iterdir()
    if file.suffix.lower() in [".txt", ".md", ".pdf", ".docx"]
]

print("\nCareer notes found:", len(career_note_files))

for file in career_note_files:
    print("-", file.name)


# ------------------------------------------------------------
# 5. LOAD CAREER NOTE TEXT
# ------------------------------------------------------------

from src.parsing.loader import load_text

career_note_texts = []

for file in career_note_files:

    text = load_text(file)

    career_note_texts.append(text)

    print("\nLoaded:", file.name)
    print("Characters:", len(text))


# ============================================================
# PART C - CREATE EMBEDDINGS
# ============================================================

print("\n" + "=" * 60)
print("CREATING JOB EMBEDDINGS")
print("=" * 60)


# ------------------------------------------------------------
# 6. EMBED JOBS
# ------------------------------------------------------------

# For Notebook 01, test only a few jobs
sample_job_texts = job_texts[:3]

job_embeddings = []

for i, job_text in enumerate(sample_job_texts):

    embedding = embed_text(job_text)

    job_embeddings.append(embedding)

    print(
        f"Job {i + 1} embedding dimension:",
        len(embedding)
    )


# ------------------------------------------------------------
# 7. EMBED CAREER NOTES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CREATING CAREER NOTE EMBEDDINGS")
print("=" * 60)

career_note_embeddings = []

for i, note_text in enumerate(career_note_texts[:3]):

    embedding = embed_text(note_text)

    career_note_embeddings.append(embedding)

    print(
        f"Career note {i + 1} embedding dimension:",
        len(embedding)
    )


# ============================================================
# PART D - TEST COSINE SIMILARITY
# ============================================================

print("\n" + "=" * 60)
print("COSINE SIMILARITY TEST")
print("=" * 60)


# Compare two job embeddings
if len(job_embeddings) >= 2:

    job_similarity = cosine_similarity(
        job_embeddings[0],
        job_embeddings[1]
    )

    print("\nJob 1 vs Job 2 similarity:")
    print(job_similarity)


# Compare a job with a career note
if job_embeddings and career_note_embeddings:

    job_note_similarity = cosine_similarity(
        job_embeddings[0],
        career_note_embeddings[0]
    )

    print("\nJob 1 vs Career Note 1 similarity:")
    print(job_note_similarity)


print("\n" + "=" * 60)
print("EMBEDDING EXPERIMENT COMPLETED")
print("=" * 60)

Project folder: c:\Users\koppa\SmartHire-GenAI
Embedding functions imported successfully.

Jobs file:
C:\Users\koppa\SmartHire-GenAI\data\jobs\naukri_com-job_sample.csv

Number of jobs: 22000
Job columns:
['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']

Jobs converted to text: 22000

SAMPLE JOB TEXT
Walkin Data Entry Operator (night Shift)
ITES
Job Description   Send me Jobs like this Qualifications: - == > 10th To Graduation & Any Skill: - == > Basic Computer Knowledge Job Requirement : - == > System or Laptop Type of job: - == > Full Time or Part time Languages : - == > Tamil & English. Experience : - == > Freshers & Experience payment details: - 1 form per day 5/- 10 form per day 50/- 100 form per day 500/- monthly you can earn 15000/- per month Selection Process: - == > Easy Selection Process,So What Are You Waiting For? Apply Now & Grab Bes

In [25]:
# ============================================================
# NOTEBOOK 01 - EMBEDDINGS
# Jobs + Career Notes + Career Guides
# ============================================================

import sys
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# 2. IMPORT PROJECT FUNCTIONS
# ------------------------------------------------------------

from src import config
from src.parsing.loader import load_text
from src.search.embed import embed_text, cosine_similarity


# ============================================================
# 3. LOAD JOBS
# ============================================================

jobs_file = config.JOBS_CSV

jobs_df = pd.read_csv(jobs_file)

print("Total jobs:", len(jobs_df))


# ------------------------------------------------------------
# CREATE TEXT FOR EACH JOB
# ------------------------------------------------------------

job_texts = []

for _, row in jobs_df.iterrows():

    parts = []

    for column in config.JOB_TEXT_COLUMNS:

        if column in jobs_df.columns:

            value = row[column]

            if pd.notna(value):
                parts.append(str(value))

    job_texts.append("\n".join(parts))


# ------------------------------------------------------------
# EMBED ALL JOBS
# ------------------------------------------------------------

job_embeddings = []

for i, text in enumerate(job_texts):

    embedding = embed_text(text)

    job_embeddings.append(embedding)

    print(
        f"Job {i + 1}/{len(job_texts)} embedded "
        f"(dimension: {len(embedding)})"
    )


# ============================================================
# 4. LOAD ALL CAREER NOTES
#    INCLUDING career_guides AND ITS SUBFOLDERS
# ============================================================

career_notes_folder = config.CAREER_NOTES_DIR

career_note_files = [
    file
    for file in career_notes_folder.rglob("*")
    if file.is_file()
    and file.suffix.lower() in [
        ".txt",
        ".md",
        ".pdf",
        ".docx"
    ]
]

print("\nTotal career notes:", len(career_note_files))

for file in career_note_files:
    print("-", file.relative_to(career_notes_folder))


# ------------------------------------------------------------
# LOAD AND EMBED ALL CAREER NOTES
# ------------------------------------------------------------

career_note_embeddings = {}

for file in career_note_files:

    text = load_text(file)

    embedding = embed_text(text)

    # Use relative path as key so files with the same name
    # in different folders don't overwrite each other
    relative_path = str(
        file.relative_to(career_notes_folder)
    )

    career_note_embeddings[relative_path] = embedding

    print(
        f"Career note embedded: {relative_path} "
        f"(dimension: {len(embedding)})"
    )


# ============================================================
# 5. COSINE SIMILARITY TEST
# ============================================================

if len(job_embeddings) >= 2:

    job_similarity = cosine_similarity(
        job_embeddings[0],
        job_embeddings[1]
    )

    print("\nJob 1 vs Job 2 similarity:", job_similarity)


# Compare first job with first career note
if job_embeddings and career_note_embeddings:

    first_note_embedding = next(
        iter(career_note_embeddings.values())
    )

    job_note_similarity = cosine_similarity(
        job_embeddings[0],
        first_note_embedding
    )

    print(
        "Job 1 vs Career Note similarity:",
        job_note_similarity
    )


# ============================================================
# 6. FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("EMBEDDING COMPLETED")
print("=" * 60)

print("Jobs embedded:", len(job_embeddings))
print(
    "Career notes + career guides embedded:",
    len(career_note_embeddings)
)
print("Embedding model:", config.EMBED_MODEL)
print("Embedding dimension:", config.EMBED_DIM)

Total jobs: 22000
Job 1/22000 embedded (dimension: 768)
Job 2/22000 embedded (dimension: 768)
Job 3/22000 embedded (dimension: 768)
Job 4/22000 embedded (dimension: 768)
Job 5/22000 embedded (dimension: 768)
Job 6/22000 embedded (dimension: 768)
Job 7/22000 embedded (dimension: 768)
Job 8/22000 embedded (dimension: 768)
Job 9/22000 embedded (dimension: 768)
Job 10/22000 embedded (dimension: 768)
Job 11/22000 embedded (dimension: 768)
Job 12/22000 embedded (dimension: 768)
Job 13/22000 embedded (dimension: 768)
Job 14/22000 embedded (dimension: 768)
Job 15/22000 embedded (dimension: 768)
Job 16/22000 embedded (dimension: 768)
Job 17/22000 embedded (dimension: 768)
Job 18/22000 embedded (dimension: 768)
Job 19/22000 embedded (dimension: 768)
Job 20/22000 embedded (dimension: 768)
Job 21/22000 embedded (dimension: 768)
Job 22/22000 embedded (dimension: 768)
Job 23/22000 embedded (dimension: 768)
Job 24/22000 embedded (dimension: 768)
Job 25/22000 embedded (dimension: 768)
Job 26/22000 emb

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [27]:
# ============================================================
# NOTEBOOK 01 - EMBEDDINGS
# Jobs + Career Notes + Career Guides
# ============================================================

import sys
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# 2. IMPORT PROJECT FUNCTIONS
# ------------------------------------------------------------

from src import config
from src.parsing.loader import load_text
from src.search.embed import embed_text, cosine_similarity


# ============================================================
# 3. LOAD JOBS
# ============================================================

jobs_file = config.JOBS_CSV
jobs_df = pd.read_csv(jobs_file)

print("Total jobs in CSV:", len(jobs_df))

# Use ONLY the first 10 jobs for testing
jobs_df = jobs_df.head(10).copy()

print("Jobs selected for embedding:", len(jobs_df))


# ------------------------------------------------------------
# CREATE TEXT FOR EACH JOB
# ------------------------------------------------------------

job_texts = []

for _, row in jobs_df.iterrows():

    parts = []

    for column in config.JOB_TEXT_COLUMNS:

        if column in jobs_df.columns:

            value = row[column]

            if pd.notna(value):
                parts.append(str(value))

    job_texts.append("\n".join(parts))


# ------------------------------------------------------------
# EMBED ONLY 10 JOBS
# ------------------------------------------------------------

job_embeddings = []

for i, text in enumerate(job_texts):

    embedding = embed_text(text)

    job_embeddings.append(embedding)

    print(
        f"Job {i + 1}/{len(job_texts)} embedded "
        f"(dimension: {len(embedding)})"
    )


# ============================================================
# 4. LOAD ALL CAREER NOTES
#    INCLUDING career_guides AND ITS SUBFOLDERS
# ============================================================

career_notes_folder = config.CAREER_NOTES_DIR

career_note_files = [

    file

    for file in career_notes_folder.rglob("*")

    if file.is_file()

    and file.suffix.lower() in [
        ".txt",
        ".md",
        ".pdf",
        ".docx"
    ]
]

print("\nTotal career notes:", len(career_note_files))

for file in career_note_files:

    print("-", file.relative_to(career_notes_folder))


# ------------------------------------------------------------
# LOAD AND EMBED ALL CAREER NOTES
# ------------------------------------------------------------

career_note_embeddings = {}

for file in career_note_files:

    text = load_text(file)

    embedding = embed_text(text)

    # Use relative path as key so files with the same name
    # in different folders don't overwrite each other

    relative_path = str(
        file.relative_to(career_notes_folder)
    )

    career_note_embeddings[relative_path] = embedding

    print(
        f"Career note embedded: {relative_path} "
        f"(dimension: {len(embedding)})"
    )


# ============================================================
# 5. COSINE SIMILARITY TEST
# ============================================================

if len(job_embeddings) >= 2:

    job_similarity = cosine_similarity(
        job_embeddings[0],
        job_embeddings[1]
    )

    print(
        "\nJob 1 vs Job 2 similarity:",
        job_similarity
    )


# Compare first job with first career note

if job_embeddings and career_note_embeddings:

    first_note_embedding = next(
        iter(career_note_embeddings.values())
    )

    job_note_similarity = cosine_similarity(
        job_embeddings[0],
        first_note_embedding
    )

    print(
        "Job 1 vs Career Note similarity:",
        job_note_similarity
    )


# ============================================================
# 6. FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("EMBEDDING COMPLETED")
print("=" * 60)

print("Jobs embedded:", len(job_embeddings))

print(
    "Career notes + career guides embedded:",
    len(career_note_embeddings)
)

print("Embedding model:", config.EMBED_MODEL)
print("Embedding dimension:", config.EMBED_DIM)

Total jobs in CSV: 22000
Jobs selected for embedding: 10
Job 1/10 embedded (dimension: 768)
Job 2/10 embedded (dimension: 768)
Job 3/10 embedded (dimension: 768)
Job 4/10 embedded (dimension: 768)
Job 5/10 embedded (dimension: 768)
Job 6/10 embedded (dimension: 768)
Job 7/10 embedded (dimension: 768)
Job 8/10 embedded (dimension: 768)
Job 9/10 embedded (dimension: 768)
Job 10/10 embedded (dimension: 768)

Total career notes: 12
- data_analyst_roadmap.md
- resume_writing_tips.md
- Career_Guides\AI_Engineer_Roadmap_2025.md
- Career_Guides\Backend_Developer_Roadmap.md
- Career_Guides\Career_Roadmap_for_Freshers.md
- Career_Guides\Data_Analyst_Role_Guide.md
- Career_Guides\Frontend_Developer_Roadmap.md
- Career_Guides\Full_Stack_Developer_Role_Guide.md
- Career_Guides\GenAI_Developer_Roadmap.md
- Career_Guides\Interview_Preparation_Guide.md
- Career_Guides\ML_Engineer_Roadmap.md
- Career_Guides\Resume_and_Portfolio_Guide.md
Career note embedded: data_analyst_roadmap.md (dimension: 768)
Car

In [28]:
# ============================================================
# EMBEDDINGS FOR ALL 22,000 JOBS
# Using Sentence Transformers (NOT GEMINI)
# ============================================================

import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer


# ============================================================
# 1. PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 2. IMPORT PROJECT CONFIG
# ============================================================

from src import config


# ============================================================
# 3. LOAD ALL JOBS
# ============================================================

jobs_file = config.JOBS_CSV

jobs_df = pd.read_csv(jobs_file)

print("Total jobs:", len(jobs_df))


# ============================================================
# 4. CREATE TEXT FOR EACH JOB
# ============================================================

job_texts = []

for _, row in jobs_df.iterrows():

    parts = []

    for column in config.JOB_TEXT_COLUMNS:

        if column in jobs_df.columns:

            value = row[column]

            if pd.notna(value):
                parts.append(str(value))

    job_texts.append("\n".join(parts))


print("Job texts created:", len(job_texts))


# ============================================================
# 5. LOAD ANOTHER EMBEDDING MODEL
#    NO GEMINI API KEY USED
# ============================================================

print("\nLoading Sentence Transformer model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")


# ============================================================
# 6. CREATE EMBEDDINGS FOR ALL JOBS
# ============================================================

print("\nCreating embeddings for all jobs...")

job_embeddings = model.encode(
    job_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

job_embeddings = np.asarray(
    job_embeddings,
    dtype=np.float32
)


# ============================================================
# 7. CHECK EMBEDDINGS
# ============================================================

print("\n" + "=" * 60)
print("JOB EMBEDDINGS CREATED")
print("=" * 60)

print("Number of jobs:", len(job_embeddings))
print("Embedding shape:", job_embeddings.shape)
print("Embedding dimension:", job_embeddings.shape[1])

c:\Users\koppa\SmartHire-GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total jobs: 22000
Job texts created: 22000

Loading Sentence Transformer model...


c:\Users\koppa\SmartHire-GenAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\koppa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3100.96it/s]


Model loaded successfully.

Creating embeddings for all jobs...


Batches: 100%|██████████| 688/688 [13:13<00:00,  1.15s/it]  



JOB EMBEDDINGS CREATED
Number of jobs: 22000
Embedding shape: (22000, 384)
Embedding dimension: 384
